In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")  # Ignore warnings for cleaner output

# Set plotting style for better aesthetics
sns.set_style("whitegrid")

print("Libraries imported successfully!")

## Accelerometer Data

In [ ]:
# Load the dataset from the CSV file
try:
    df = pd.read_sas('PAXMIN_H.xpt')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Error: Make sure dataset is in the same directory as this notebook.")

# Display the first 5 rows to get a feel for the data
df.head()

In [ ]:
df = df[['SEQN','PAXDAYM','PAXDAYWM','PAXSSNMP','PAXMXM','PAXMXM','PAXMXM','PAXMTSM','PAXPREDM']]

In [ ]:
df.info()

In [ ]:
# Decode bytes columns
df['PAXDAYM'] = df['PAXDAYM'].str.decode('utf-8')
df['PAXDAYWM'] = df['PAXDAYWM'].str.decode('utf-8')
df['PAXPREDM'] = df['PAXPREDM'].str.decode('utf-8')

In [ ]:
type(df['PAXPREDM'].iloc[0])

In [ ]:
print("Unique values in 'PAXPREDM' BEFORE cleaning:", df['PAXPREDM'].unique())

In [ ]:
# Step 0: Exclude each participant's first recorded day
df = (
    df
    .sort_values(['SEQN', 'PAXDAYM'])
    .groupby('SEQN')
    .apply(lambda g: g[g['PAXDAYM'] != '1'])
    .reset_index(drop=True)
)

In [ ]:
# Sort data properly
df = df.sort_values(['SEQN','PAXDAYWM','PAXSSNMP']).copy()

# Compute difference between successive sample counts
df['delta_samples'] = df.groupby(['SEQN','PAXDAYWM'])['PAXSSNMP'].diff()

# Each minute = 4800 samples
SAMPLES_PER_MIN = 4800

# Minutes since start of day
df['minute_increment'] = (df['delta_samples'] / SAMPLES_PER_MIN).fillna(0)

# Cumulative minute count starting at 0:00
df['minute_of_day'] = df.groupby(['SEQN','PAXDAYWM'])['minute_increment'].cumsum().astype(int)

# Hour of day
df['hour_of_day'] = df['minute_of_day'] // 60

# Create time-of-day bins
df['time_period'] = pd.cut(
    df['hour_of_day'],
    bins=[-1, 4, 11, 16, 20, 23],
    labels=['night', 'morning', 'afternoon', 'evening', 'night_2'],
    right=True
)

# Merge the two night categories
df['time_period'] = df['time_period'].replace({'night_2': 'night'})

In [ ]:
# Remove rows with non wear or unknown wear
print("Rows in dataset BEFORE cleaning:", len(df))
print("Participants in dataset BEFORE cleaning:",df['SEQN'].nunique())
df_cleaned = df[(df['PAXPREDM'] == "1") | (df['PAXPREDM'] == "2")]
print("Unique values in 'PAXPREDM' AFTER cleaning:", df_cleaned['PAXPREDM'].unique())
print("Rows in dataset AFTER PAXPREDM cleaning:", len(df_cleaned))

# Remove rows with -0.01 value in PAXMTSM (value could not be computed)
df_cleaned = df_cleaned[df_cleaned['PAXMTSM'] != -0.01]

print("Rows in dataset AFTER PAXMTSM cleaning:", len(df_cleaned))
print("Participants in dataset AFTER cleaning:",df_cleaned['SEQN'].nunique())

In [ ]:
# Step 1: Filter out days with < 16 hours wear time
valid_days = (
    df_cleaned
    .groupby(['SEQN', 'PAXDAYM'])
    .filter(lambda g: len(g) >= 16 * 60)
)
# Step 2: Keep only participants with ≥ 4 valid days
final_df_cleaned = (
    valid_days
    .groupby('SEQN')
    .filter(lambda g: g['PAXDAYM'].nunique() >= 4)
)
print("Participants retained after filtering:", final_df_cleaned['SEQN'].nunique())

In [ ]:
print("Rows in dataset AFTER cleaning:", len(final_df_cleaned))

In [ ]:
print("Participants in dataset AFTER cleaning:",final_df_cleaned['SEQN'].nunique())

In [ ]:
final_df_cleaned.to_pickle('accelerometer_data_start.pkl')

## Load accelerometer data

In [ ]:
final_df_cleaned = pd.read_pickle('accelerometer_data_start.pkl')

## Demographics Data

In [ ]:
# Load the dataset from the CSV file
try:
    demo_df = pd.read_sas('DEMO_H.xpt')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Error: Make sure dataset is in the same directory as this notebook.")

# Display the first 5 rows to get a feel for the data
demo_df.head()

In [ ]:
print("Participants in dataset BEFORE cleaning:",demo_df['SEQN'].nunique())

In [ ]:
demo_df = demo_df[['SEQN','RIAGENDR','RIDAGEYR','RIDRETH3','DMDCITZN','DMDEDUC2','DMDMARTL','DMDHHSIZ','INDHHIN2','INDFMPIR']]

In [ ]:
demo_df = demo_df.rename(columns={'RIAGENDR':"SEX",'RIDAGEYR':"AGE",'RIDRETH3':"RACE",'DMDCITZN':"CITIZENSHIP",'DMDEDUC2':"EDUCATION",'DMDMARTL':"MARITAL_STATUS",'DMDHHSIZ':"HH_NUMBER",'INDHHIN2':"HH_INCOME",'INDFMPIR':'RATIO_POVERTY'})
demo_df.head()

In [ ]:
demo_df.info()

In [ ]:
# Remove rows with refusal, don't know, or missing values in citizenship
demo_df = demo_df[~demo_df['CITIZENSHIP'].isin([7, 9]) & demo_df['CITIZENSHIP'].notna()]
# Remove rows with refused or don't know marital status
demo_df = demo_df[~demo_df['MARITAL_STATUS'].isin([77, 99])]
# Remove rows with refused or don't know education
demo_df = demo_df[~demo_df['EDUCATION'].isin([7, 9]) & demo_df['EDUCATION'].notna()]
# Remove rows with refusal, don't know, or missing values in income
demo_df = demo_df[~demo_df['HH_INCOME'].isin([77, 99]) & demo_df['HH_INCOME'].notna()]
# Remove rows with mssing poverty ratio
demo_df = demo_df[demo_df['RATIO_POVERTY'].notna()]

In [ ]:
demo_df.info()

In [ ]:
# Only keep people 20+
demo_df = demo_df[demo_df['AGE'] >= 20]

In [ ]:
print("Participants in dataset AFTER cleaning:",demo_df['SEQN'].nunique())

In [ ]:
demo_df.to_csv('demographics_data.csv')

## Health Status Data

In [ ]:
# Load the dataset from the CSV file
try:
    hsq_df = pd.read_sas('DPQ_H.xpt')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Error: Make sure dataset is in the same directory as this notebook.")

# Display the first 5 rows to get a feel for the data
hsq_df.head()

In [ ]:
print("Participants in dataset BEFORE cleaning:",hsq_df['SEQN'].nunique())

In [ ]:
hsq_df = hsq_df[['SEQN','DPQ010','DPQ020','DPQ030','DPQ040','DPQ050','DPQ060','DPQ070','DPQ080','DPQ090']]
# hsq_df = hsq_df.rename(columns={'DPQ010':'INTEREST','DPQ020':'DEPRESSED','DPQ030':'SLEEP_TROUBLE','DPQ040':'TIRED','DPQ050':'FOOD_TROUBLE','DPQ060':'POOR_SELF_IMAGE','DPQ090':'SUICIDAL_THOUGHTS','DPQ100':'LIFE_DIFFICULTY'})

In [ ]:
hsq_df.info()

In [ ]:
# Remove refused, don't know and missing
hsq_df = hsq_df[~hsq_df['DPQ010'].isin([7, 9]) & hsq_df['DPQ020'].notna()]
hsq_df = hsq_df[~hsq_df['DPQ020'].isin([7, 9]) & hsq_df['DPQ020'].notna()]
hsq_df = hsq_df[~hsq_df['DPQ030'].isin([7, 9]) & hsq_df['DPQ030'].notna()]
hsq_df = hsq_df[~hsq_df['DPQ040'].isin([7, 9]) & hsq_df['DPQ040'].notna()]
hsq_df = hsq_df[~hsq_df['DPQ050'].isin([7, 9]) & hsq_df['DPQ050'].notna()]
hsq_df = hsq_df[~hsq_df['DPQ060'].isin([7, 9]) & hsq_df['DPQ060'].notna()]
hsq_df = hsq_df[~hsq_df['DPQ070'].isin([7, 9]) & hsq_df['DPQ070'].notna()]
hsq_df = hsq_df[~hsq_df['DPQ080'].isin([7, 9]) & hsq_df['DPQ080'].notna()]
hsq_df = hsq_df[~hsq_df['DPQ090'].isin([7, 9]) & hsq_df['DPQ090'].notna()]

In [ ]:
print("Participants in dataset AFTER cleaning:",hsq_df['SEQN'].nunique())

In [ ]:
hsq_df['Total_Depression_Score'] = (hsq_df['DPQ010'] + hsq_df['DPQ020'] + hsq_df['DPQ030'] + hsq_df['DPQ040'] +
                                   hsq_df['DPQ050'] + hsq_df['DPQ060'] + hsq_df['DPQ070'] + hsq_df['DPQ080'] + hsq_df['DPQ090'])

In [ ]:
demo_df = demo_df.merge(hsq_df[['SEQN', 'Total_Depression_Score']], on='SEQN', how='left')
demo_df.head()

In [ ]:
# drop nan rows
demo_df = demo_df.dropna()

In [ ]:
print("Participants in dataset AFTER cleaning:",demo_df['SEQN'].nunique())

In [ ]:
# Resave demographics
demo_df.to_csv('demographics_data.csv')

## Reduce datasets

In [ ]:
# Check for fully duplicated rows in accelerometer
total_duplicates = final_df_cleaned.duplicated().sum()
print(f"Total fully duplicated rows: {total_duplicates}")

# # Check for duplicates based on interview identifiers, as done in the report [cite: 1079]
# # Note: The report uses "No_Pation" but the CSV has "No_Pation". We'll use the correct column name.
# interview_duplicates = final_df_cleaned.duplicated(subset=["ADM_RNO1"]).sum()
# print(f"Rows with the same interviewee: {interview_duplicates}")

In [ ]:
# Check for fully duplicated rows in demographics
total_duplicates = demo_df.duplicated().sum()
print(f"Total fully duplicated rows: {total_duplicates}")

# Check for duplicates based on survey id number
survey_duplicates = demo_df.duplicated(subset=["SEQN"]).sum()
print(f"Rows with the same participant: {survey_duplicates}")

In [ ]:
# Convert to sets and get intersection
df_parts = final_df_cleaned['SEQN'].unique()
demo_parts = demo_df['SEQN'].unique()

common_values = set(df_parts) & set(demo_parts)

print(len(common_values))

In [ ]:
final_accel_df = final_df_cleaned[final_df_cleaned['SEQN'].isin(common_values)]
final_demo_df = demo_df[demo_df['SEQN'].isin(common_values)]

In [ ]:
final_demo_df = demo_df[demo_df['SEQN'].isin(common_values)]

In [ ]:
final_accel_df.info()

In [ ]:
final_accel_df['SEQN'].nunique()

In [ ]:
final_demo_df.info()

In [ ]:
# Save the final dataframes
final_demo_df.to_csv('demographics_data.csv')
final_accel_df.to_pickle('accelerometer_data.pkl')

In [ ]:
len(final_demo_df[final_demo_df['Total_Depression_Score'] >= 10])

In [ ]:
hsq_df.head()

## Accelerometer Features

### Physical Activity

In [ ]:
final_accel_df = pd.read_pickle('accelerometer_data.pkl')

In [ ]:
# For each row in the accelerometer dataset include if it is
# sedentary, light, mod, vigorous

# Sedentary: < 4 MIMS/min

# Light activity: 4–10 MIMS/min

# Moderate activity: 10–20 MIMS/min

# Vigorous activity: > 20 MIMS/min

conditions = [
    final_accel_df['PAXMTSM'] < 10.6,
    (final_accel_df['PAXMTSM'] >= 10.6) & (final_accel_df['PAXMTSM'] < 19.6),
    (final_accel_df['PAXMTSM'] >= 19.6) ]

choices = ['Sedentary', 'Light', 'Moderate']

final_accel_df['ACTIVITY'] = np.select(conditions, choices, default=None)

In [ ]:
# Keep only valid wear
df = final_accel_df[final_accel_df['PAXPREDM'] == '1']

# Count minutes per day per activity
daily_activity = (
    df.groupby(['SEQN', 'PAXDAYM', 'ACTIVITY'])
      .size()
      .reset_index(name='minutes')
)

# Now compute average minutes per day per activity per participant
avg_daily_activity = (
    daily_activity.groupby(['SEQN', 'ACTIVITY'])['minutes']
                  .mean()
                  .reset_index(name='avg_daily_minutes')
)

In [ ]:
daily_activity

In [ ]:
avg_daily_activity

In [ ]:
# Calculate avg wake wear MIMS for each participant
# First compute daily averages
daily_avg = final_accel_df[final_accel_df['PAXPREDM'] == '1'].groupby(['SEQN','PAXDAYM'])['PAXMTSM'].mean().reset_index(name='DAILY_MEAN_MIMS')

# Then average across days per participant
avg_daily_mims = daily_avg.groupby('SEQN')['DAILY_MEAN_MIMS'].mean().reset_index()

In [ ]:
# Peak 30 MIMS
df_wake = final_accel_df[final_accel_df['PAXPREDM'] == '1'].copy()
daily_peak30 = (
    df_wake
    .groupby(['SEQN', 'PAXDAYM'])['PAXMTSM']
    .apply(lambda x: x.nlargest(30).sum())
    .reset_index(name='DAILY_PEAK30_MIMS')
)
avg_peak30_mims = (
    daily_peak30.groupby('SEQN')['DAILY_PEAK30_MIMS']
    .mean()
    .reset_index(name='AVG_PEAK30_MIMS')
)

In [ ]:
activity_summary = (
    df_wake.groupby(['SEQN','PAXDAYM','time_period'])['PAXMTSM']
    .mean()
    .reset_index(name='avg_mims')
)

In [ ]:
import itertools

# Compute total avg daily MIMS per participant
daily_total = (
    activity_summary.groupby(['SEQN','PAXDAYM'])['avg_mims']
    .sum()
    .reset_index(name='daily_total_mims')
)

# Merge with activity_summary
activity_norm = activity_summary.merge(daily_total, on=['SEQN','PAXDAYM'])

# Compute proportion of MIMS per time period
activity_norm['mims_fraction'] = activity_norm['avg_mims'] / activity_norm['daily_total_mims']

# Now you can average across days for each participant and time period
participant_summary = (
    activity_norm.groupby(['SEQN','time_period'])['mims_fraction']
    .mean()
    .reset_index()
)

# All unique participants and time periods
participants = activity_norm['SEQN'].unique()
time_periods = activity_norm['time_period'].unique()

# Create all possible combinations
all_combinations = pd.DataFrame(list(itertools.product(participants, time_periods)), columns=['SEQN','time_period'])

# Merge with your computed participant_summary
participant_summary = all_combinations.merge(participant_summary, on=['SEQN','time_period'], how='left')

# Fill missing fractions with 0
participant_summary['mims_fraction'] = participant_summary['mims_fraction'].fillna(0)

participant_summary

In [ ]:
participant_summary

### Sleep

In [ ]:
# Filter sleep minutes
sleep_minutes = final_accel_df[final_accel_df['PAXPREDM'] == '2']

# Daily sleep duration (in minutes)
daily_sleep = (
    sleep_minutes.groupby(['SEQN', 'PAXDAYM'])['PAXMTSM']
    .count()
    .reset_index(name='SLEEP_DURATION_MIN')
)

# Average sleep duration per participant
avg_sleep_duration = (
    daily_sleep.groupby('SEQN')['SLEEP_DURATION_MIN']
    .mean()
    .reset_index(name='AVG_SLEEP_DURATION_MIN')
)

# Sleep duration variability per participant (standard deviation)
sleep_variability = (
    daily_sleep.groupby('SEQN')['SLEEP_DURATION_MIN']
    .std()
    .reset_index(name='SLEEP_DURATION_SD')
)

# Merge into one dataframe
sleep_summary = avg_sleep_duration.merge(sleep_variability, on='SEQN')

sleep_summary['AVG_SLEEP_HOURS'] = sleep_summary['AVG_SLEEP_DURATION_MIN'] / 60
sleep_summary['SLEEP_SD_HOURS'] = sleep_summary['SLEEP_DURATION_SD'] / 60

sleep_summary.head()

In [ ]:
df_sleep = final_accel_df[(final_accel_df['time_period'] == 'evening') | (final_accel_df['time_period'] == 'night') | (final_accel_df['time_period'] == 'morning')].copy()
df_sleep['sleeping'] = df_sleep['PAXPREDM'].astype(str) == '2'

In [ ]:
from skdh.sleep import NumberWakeBouts

# Convert day column to integer if needed
df_sleep['PAXDAYM'] = df_sleep['PAXDAYM'].astype(int)

# Prepare output list
sleep_summary_list = []

# Loop over participants
for pid, df_pid in df_sleep.groupby('SEQN'):
    # Loop over days
    for day, df_day in df_pid.groupby('PAXDAYM'):
        # scikit-digital-health expects 1D activity vector
        # We'll use triaxial magnitude or a single axis
        # Here, we can use the total MIMS (PAXMTSM)
        activity = df_day['sleeping'].to_numpy()

        # Compute Number of Wake Bouts
        wake_bouts = NumberWakeBouts().predict(activity)
        
        sleep_summary_list.append({
            'SEQN': pid,
            'PAXDAYM': day,
            'wake_bouts': wake_bouts
        })

# Convert to dataframe
sleep_day_df = pd.DataFrame(sleep_summary_list)

# --- Compute per-participant averages and variability ---

participant_sleep_summary = (
    sleep_day_df.groupby('SEQN')
    .agg(
        avg_wake_bouts=('wake_bouts', 'mean')
    )
    .reset_index()
)

sleep_summary['Avg_wake_bouts'] = participant_sleep_summary['avg_wake_bouts']

print(sleep_summary.head())

In [ ]:
participant_summary

In [ ]:
activity_wide = avg_daily_activity.pivot(index='SEQN', columns='ACTIVITY', values='avg_daily_minutes').reset_index()
activity_period_wide = participant_summary.pivot(index='SEQN', columns='time_period', values='mims_fraction').reset_index()

# Merge with average daily MIMS and peak 30 daily MIMS
summary = activity_wide.merge(avg_daily_mims, on='SEQN')
summary = summary.merge(activity_period_wide, on='SEQN')
summary = summary.merge(avg_peak30_mims, on='SEQN')

# Merge sleep time
summary = summary.merge(sleep_summary, on='SEQN')

In [ ]:
# Merge with demographics
summary_demo_df = final_demo_df.merge(summary, on='SEQN')

In [ ]:
summary_demo_df

In [ ]:
summary_demo_df = summary_demo_df.dropna()

In [ ]:
len(summary_demo_df)

In [ ]:
# Save the final dataframes
summary_demo_df.to_csv('demographics_data.csv')
final_accel_df.to_pickle('accelerometer_data.pkl')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('demographics_data.csv')

In [ ]:
df['Depression_Category'] = df["Total_Depression_Score"].apply(lambda x: 1 if x >= 10 else 0)

In [ ]:
sensitive_features = ['SEX', 'AGE', 'RACE', 'CITIZENSHIP', 'EDUCATION', 'MARITAL_STATUS','HH_NUMBER', 'HH_INCOME', 'RATIO_POVERTY']

In [ ]:
for feat in sensitive_features:
    if feat == 'AGE':
        # Create age groups
        bins = [20, 30, 40, 50, 60, 70, 120]
        labels = ["20-29", "30-39", "40-49", "50-59", "60-69", "70+"]
        df["AGE"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)
    if feat == 'RATIO_POVERTY':
        bins=[0,1,2,3,4,5]
        labels=[0,1,2,3,4]
        df["RATIO_POVERTY"] = pd.cut(df["RATIO_POVERTY"], bins=bins, labels=labels, right=False)

    # Compute raw counts
    counts_race = (
        df.groupby(feat)["Depression_Category"]
        .value_counts()
        .unstack(fill_value=0)
    )

    counts_race.columns = ["No depression", "Moderate-Severe depression"]

    # Plot
    counts_race.plot(kind="bar", stacked=False, figsize=(9,5),
                    color=["steelblue", "darkorange"])

    plt.ylabel("Count",fontsize=14)
    plt.xlabel(f"{feat}",fontsize=14)
    plt.title(f"Depression Category by {feat}")

    plt.xticks(
        ticks=range(len(counts_race.index)),
        # labels=['Mexican American','Other Hispanic','Non-Hispanic White',
        #         'Non-Hispanic Black','Non-Hispanic Asian','Other'],
        rotation=20
    ,fontsize=14)
    plt.yticks(fontsize=14)

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
demo_cols = ["SEX", "AGE", "RACE", "CITIZENSHIP", "EDUCATION",
             "MARITAL_STATUS", "HH_NUMBER", "HH_INCOME", "RATIO_POVERTY"]

outcome = "Depression_Category"

In [ ]:
combined_counts = []

for col in demo_cols:
    temp = (
        df.groupby(col)[outcome]
          .value_counts()
          .unstack(fill_value=0)
          .reset_index()
    )
    temp.insert(0, "Feature", col)  # add column name for clarity
    combined_counts.append(temp)

combined_counts_df = pd.concat(combined_counts, ignore_index=True)
combined_counts_df